In [38]:
import socket
import threading
import time

In [3]:
SERVICE_MAP = {
    21: "FTP",
    22: "SSH",
    25: "SMTP",
    80: "HTTP",
    443: "HTTPS",
    23: "Telnet",
    53: "DNS",
    110: "POP3",
    143: "IMAP"
}

In [17]:
def resolve_host(target):
    try:
        # Check if already an IP address
        socket.inet_aton(target)
        return target
    except socket.error:
        # Try to resolve hostname
        try:
            ip = socket.gethostbyname(target)
            return ip
        except socket.gaierror:
            print(f"Error: Cannot resolve hostname '{target}'")
            return None

In [18]:
print( resolve_host("www.google.com"))

172.217.169.228


In [19]:
print(resolve_host("172.217.169.228"))

172.217.169.228


In [34]:
def parse_ports(port_spec):
    
    ports = []

    # Split by comma for multiple specifications
    parts = port_spec.split(',')

    for part in parts:
        part = part.strip()

        # Check if it's a range
        if '-' in part:
            try:
                start, end = part.split('-')
                start = int(start.strip())
                end = int(end.strip())
                

                if start > end or start < 1 or end > 65535:
                    print(f"Invalid port range: {part}")
                    continue

                ports.extend(range(start, end + 1))
            except ValueError:
                print(f"Invalid port range format: {part}")
                continue
        else:
            # Single port
            try:
                port = int(part)
                if 1 <= port <= 65535:
                    ports.append(port)
                else:
                    print(f"Invalid port number: {port}")
            except ValueError:
                print(f"Invalid port: {part}")
                continue

    # Remove duplicates and sort
    ports = sorted(list(set(ports)))
    return ports

In [35]:
def tcp_scan(ip, port, timeout=2):
    try:
        # Create TCP socket
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(timeout)

        # Attempt connection
        result = sock.connect_ex((ip, port))
        sock.close()

        # Return True if connection successful (port open)
        return result == 0
    except socket.error:
        return False

def scan_port_wrapper(args):
    ip, port, timeout = args
    is_open = tcp_scan(ip, port, timeout)
    service = SERVICE_MAP.get(port, "Unknown")
    return (port, is_open, service)

In [50]:
def display_results(target, ip, ports, open_ports, scan_time):
    
    print(f"\nTarget: {target} ({ip})")

    # Format port list for display
    if len(ports) <= 10:
        port_list = ','.join(map(str, ports))
    else:
        port_list = f"{ports[0]}-{ports[-1]}"

    print(f"Scanning ports: {port_list}")

    if open_ports:
        print("\nOpen ports found:")
        for port, service in open_ports:
            print(f"  {port}/tcp    {service}")
    else:
        print("\nNo open ports found")

    print(f"\nScan completed in {scan_time:.5f} seconds")

In [51]:


def scan_target(target, port_spec):
    
    if not target:
        print("Error: No target specified")
        return

    # Resolve hostname to IP
    ip = resolve_host(target)
    if not ip:
        return

    if not port_spec:
        print("Error: No ports specified")
        return

    # Parse ports
    ports = parse_ports(port_spec)

    if not ports:
        print("Error: No valid ports to scan")
        return

    print(f"\nStarting scan of {len(ports)} ports...")

    # Record start time
    start_time = time.time()

   
    open_ports = []
    threads = []

    def thread_task(ip, port, timeout):
        port, is_open, service = scan_port_wrapper((ip, port, timeout))
        if is_open:
            open_ports.append((port, service))

    for port in ports:
        t = threading.Thread(target=thread_task, args=(ip, port, 2))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()
    

    # Calculate scan time
    scan_time = time.time() - start_time

    # Sort open ports by port number
    open_ports.sort()

    # Display results
    display_results(target, ip, ports, open_ports, scan_time)


# Example 1: Scan a single port
print("EXAMPLE 1: Single Port Scan")
print("-" * 30)
scan_target("scanme.nmap.org", "80-200")

print("\n" + "=" * 50 + "\n")

# Example 2: Scan multiple specific ports
print("EXAMPLE 2: Multiple Ports Scan")
print("-" * 30)
scan_target("192.168.18.90", "1-189")

print("\n" + "=" * 50 + "\n")

# Example 3: Scan a range of ports
print("EXAMPLE 3: Port Range Scan")
print("-" * 30)
scan_target("scanme.nmap.org", "20-90")

print("\n" + "=" * 50 + "\n")


EXAMPLE 1: Single Port Scan
------------------------------

Starting scan of 121 ports...

Target: scanme.nmap.org (45.33.32.156)
Scanning ports: 80-200

Open ports found:
  80/tcp    HTTP

Scan completed in 2.00586 seconds


EXAMPLE 2: Multiple Ports Scan
------------------------------

Starting scan of 189 ports...

Target: 192.168.18.90 (192.168.18.90)
Scanning ports: 1-189

No open ports found

Scan completed in 0.02534 seconds


EXAMPLE 3: Port Range Scan
------------------------------

Starting scan of 71 ports...

Target: scanme.nmap.org (45.33.32.156)
Scanning ports: 20-90

Open ports found:
  22/tcp    SSH
  80/tcp    HTTP

Scan completed in 2.00411 seconds


